# 1. Experiment Information
This notebook provides the standard execution pipeline for baseline two-stem (vocals/accompaniment) source separation experiments using the pretrained HT-Demucs model on Indian Folk Music datasets.
This version (2.0) supports batch processing of multiple recordings.

# 2. Google Drive Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 3. Repository Configuration

In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation")
print(f"Project Root: {PROJECT_ROOT}")

# 4. Experiment Configuration

In [ ]:
# --- EXPERIMENT CONFIGURATION ---

EXPERIMENT_ID = "EXP002"
NOTEBOOK_VERSION = "2.0"
MODEL_NAME = "HT-Demucs"
DATASET_NAME = "BIHU"
TWO_STEMS = "vocals"

# Derived Paths
DATASET_DIR = PROJECT_ROOT / "03_Datasets"
TRADITION_DIR = DATASET_DIR / DATASET_NAME
INPUT_AUDIO_DIR = TRADITION_DIR / "Audio" / "WAV"

EXPERIMENT_DIR = PROJECT_ROOT / "05_Experiments" / "Baseline_HT_Demucs" / EXPERIMENT_ID
OUTPUT_DIRECTORY = EXPERIMENT_DIR / "outputs"
LOG_DIRECTORY = EXPERIMENT_DIR / "logs"

print("Configuration Loaded:")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Dataset Dir  : {INPUT_AUDIO_DIR}")
print(f"Output Dir   : {OUTPUT_DIRECTORY}")
print(f"Log Dir      : {LOG_DIRECTORY}")

# 5. Environment Validation

In [ ]:
import torch
import sys

# 1. Check Paths
required_paths = {
    "Project Directory": PROJECT_ROOT,
    "Dataset Directory": DATASET_DIR,
    "Input Audio Directory": INPUT_AUDIO_DIR,
    "Output Directory": OUTPUT_DIRECTORY,
    "Log Directory": LOG_DIRECTORY
}

for name, path in required_paths.items():
    if not path.exists():
        raise RuntimeError(f"❌ ERROR: {name} does not exist at {path}")
    else:
        print(f"✓ {name} found.")

audio_files = sorted(list(INPUT_AUDIO_DIR.glob("*.wav")))
if not audio_files:
    raise RuntimeError(f"❌ ERROR: No WAV files found in {INPUT_AUDIO_DIR}")
print(f"✓ Found {len(audio_files)} audio files for processing.")

# 2. Check CUDA & PyTorch
print(f"✓ PyTorch Version: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("❌ ERROR: CUDA is not available. Please enable GPU in Colab.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ CUDA Available (GPU: {gpu_name})")

# 3. Check Demucs
try:
    import demucs
    print(f"✓ Demucs installed (Version: {demucs.__version__})")
    demucs_version = demucs.__version__
except ImportError:
    print("Demucs not found. Installing now...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs"], check=True)
    import demucs
    print(f"✓ Demucs installed (Version: {demucs.__version__})")
    demucs_version = demucs.__version__

print("\nEnvironment validation passed!")

# 6. HT-Demucs Execution

In [ ]:
import time
import subprocess
import datetime

print(f"Starting {MODEL_NAME} batch separation on {len(audio_files)} recordings...")

overall_start = time.time()
overall_start_dt = datetime.datetime.now()

results_log = []

for idx, audio_file in enumerate(audio_files, 1):
    print(f"\n[{idx}/{len(audio_files)}] Processing {audio_file.name}...")
    
    command = [
        sys.executable, "-m", "demucs",
        "--two-stems", TWO_STEMS,
        "-d", "cuda",
        "-o", str(OUTPUT_DIRECTORY),
        str(audio_file)
    ]
    
    start_time = time.time()
    
    # Execute Demucs
    result = subprocess.run(command)
    
    processing_duration = time.time() - start_time
    
    if result.returncode != 0:
        print(f"❌ ERROR: Demucs execution failed for {audio_file.name}")
        status = "Failed"
    else:
        print(f"✓ Demucs execution completed in {processing_duration:.2f} seconds.")
        status = "Success"
        
    results_log.append({
        "file": audio_file,
        "duration": processing_duration,
        "status": status,
        "command": command
    })

overall_end = time.time()
overall_end_dt = datetime.datetime.now()
total_duration = overall_end - overall_start
print(f"\nBatch processing completed in {total_duration:.2f} seconds.")

# 7. Output Processing

In [ ]:
import os
import shutil

generated_files = []

for res in results_log:
    if res['status'] != 'Success':
        continue
        
    audio_file = res['file']
    demucs_output_dir = OUTPUT_DIRECTORY / "htdemucs" / audio_file.stem
    
    vocals_original = demucs_output_dir / "vocals.wav"
    no_vocals_original = demucs_output_dir / "no_vocals.wav"
    
    vocals_new = demucs_output_dir / f"{audio_file.stem}_vocals.wav"
    no_vocals_new = demucs_output_dir / f"{audio_file.stem}_no_vocals.wav"
    
    if vocals_original.exists():
        vocals_original.rename(vocals_new)
        print(f"Renamed: {vocals_original.name} -> {vocals_new.name}")
        
    if no_vocals_original.exists():
        no_vocals_original.rename(no_vocals_new)
        print(f"Renamed: {no_vocals_original.name} -> {no_vocals_new.name}")
        
    generated_files.append((vocals_new, no_vocals_new))

# 8. Execution Log Generation

In [ ]:
log_file_path = LOG_DIRECTORY / "execution_log.txt"

log_content = f"""Experiment ID: {EXPERIMENT_ID}
Notebook Version: {NOTEBOOK_VERSION}
Execution Date: {overall_start_dt.strftime('%Y-%m-%d')}
Execution Time: {overall_start_dt.strftime('%H:%M:%S')} - {overall_end_dt.strftime('%H:%M:%S')}
Total Processing Duration: {total_duration:.2f} seconds
Model: {MODEL_NAME}
Dataset: {DATASET_NAME}
Total Files Processed: {len(audio_files)}
Output Directory: {OUTPUT_DIRECTORY}
GPU Name: {gpu_name}
PyTorch Version: {torch.__version__}
Demucs Version: {demucs_version}

--- File Level Details ---
"""

for res in results_log:
    log_content += f"\nFile: {res['file'].name}\n"
    log_content += f"Status: {res['status']}\n"
    log_content += f"Duration: {res['duration']:.2f}s\n"
    command_str = " ".join(res['command'])
    log_content += f"Command: {command_str}\n"

with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print(f"Execution log saved to {log_file_path}")

# 9. Result Summary

In [ ]:
import datetime

print("====================================")
print("Experiment Completed Successfully")
print(f"Experiment:\n{EXPERIMENT_ID}\n")
print(f"Input Dataset:\n{DATASET_NAME}\n")
print(f"Files Processed: {len(audio_files)}\n")
print(f"Execution Log:\n✓ execution_log.txt\n")
print(f"Total Processing Time:\n{datetime.timedelta(seconds=int(total_duration))}")
print("====================================")